# CVDL · Handwriting data collection

Collect **your own handwriting** using a camera photograph or an uploaded JPG, PNG or WebP image. The notebook suggests text, lets you correct both the line boundaries and the labels, and saves only approved samples to Google Drive.

**Start here:** choose **Runtime → Change runtime type → T4 GPU**, then run Cells 1–9 in order. A CPU also works but recognition is slower. Use a fresh Colab runtime; this notebook does not require your training notebook to have run.

- Cell 2 selects English or Chinese and the dataset folder. To change language, finish saving the current image, change Cell 2, then rerun Cells 2 and 6–9.
- Cell 7 downloads one recogniser and a small text detector. The first run takes longer; later calls reuse them. English uses the pretrained TrOCR baseline; Chinese uses the pretrained PP-OCRv5 server recogniser. Existing fine-tuned checkpoints are not required.
- Cell 9 is the collection interface. It remains interactive after the code cell finishes; Python callbacks handle the buttons.
- Capture one horizontal line or one single-column paragraph at a time. Multicolour ink is retained. Handwriting, pale ink and ruled pages can still require manual crop correction.
- Images are processed for transcription only. Correcting a label does **not** train the model immediately.

Your samples are assigned **approximately 75% of source pages to training and 25% to validation**, using a reproducible random order. Every crop from the same physical page stays together. Teammates' independent samples belong in the separately reserved `test/` folder and are not collected or randomly assigned by this notebook.


## Cell 1: Install the collection dependencies without replacing Colab's PyTorch

Run once per runtime. This uses the PyTorch versions already installed by Colab; it does not install PaddlePaddle or create a Paddle environment.

In [ ]:
# Cell 1: Install the collection dependencies without replacing Colab's PyTorch
# @title Install dependencies

import importlib.metadata
import os
import subprocess
import sys

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# Keep the existing torch/torchvision/CUDA combination supplied by Colab.
required_transformers = "5.17.0"
already_loaded = "transformers" in sys.modules
previous_version = getattr(sys.modules.get("transformers"), "__version__", None)
command = [sys.executable, "-m", "pip", "install", "--retries", "8", "--timeout", "120"]
help_text = subprocess.run([sys.executable, "-m", "pip", "install", "--help"],
                           capture_output=True, text=True, check=True).stdout
if "--resume-retries" in help_text:
    command += ["--resume-retries", "12"]
command += [f"transformers=={required_transformers}", "opencv-python-headless>=4.10,<4.12",
            "pyclipper>=1.3,<2", "shapely>=2,<3"]
subprocess.check_call(command)

if already_loaded and previous_version != required_transformers:
    raise RuntimeError("Dependencies were updated. Choose Runtime > Restart session, then run this notebook again.")
import torch
print("Dependencies ready. PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## Cell 2: Connect Google Drive and choose the collection language

Choose the language of the photograph. The same dataset can contain both languages. Keep one collection session writing to this dataset at a time.

In [ ]:
# Cell 2: Connect Google Drive and choose the collection language
# @title Dataset location and language

from pathlib import Path
from google.colab import drive

LANGUAGE_NAME = "English" # @param ["English", "Chinese"]
DATASET_FOLDER = "CVDL_Handwriting_Dataset/v1" # @param {type:"string"}

# The notebook collects your own handwriting. Teammate test data is reserved separately.
drive.mount("/content/drive")
LANGUAGE = {"English": "en", "Chinese": "zh"}[LANGUAGE_NAME]
drive_root = Path("/content/drive/MyDrive").resolve()
DATASET_ROOT = (drive_root / DATASET_FOLDER).resolve()
if DATASET_ROOT == drive_root or not DATASET_ROOT.is_relative_to(drive_root):
    raise ValueError("Choose a dataset subfolder inside My Drive, for example CVDL_Handwriting_Dataset/v1.")
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
print("Approved data will be saved to:", DATASET_ROOT)
print("Collection language:", LANGUAGE_NAME)


## Cell 3: Define image validation and safe file helpers

These helpers preserve the uploaded original, normalise EXIF orientation for cropping, validate crop coordinates, and verify files before committing them.

In [ ]:
# Cell 3: Define image validation and safe file helpers

import base64
import csv
import hashlib
import io
import json
import math
import os
import random
import threading
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image, ImageOps

SCHEMA_VERSION = 1
SPLITS = ("pending", "train", "validation", "test")
LABEL_FIELDS = ["sample_id", "image_path", "text", "language", "source_page_id",
                "capture_id", "line_index"]
DATA_LOCK = threading.RLock()


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def digest(data):
    return hashlib.sha256(data).hexdigest()


def write_verified(path, data):
    """Write a temporary file, verify it, then replace the destination."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temporary.open("wb") as stream:
            stream.write(data)
            stream.flush()
        if digest(temporary.read_bytes()) != digest(data):
            raise IOError("File verification failed; retry saving this capture.")
        os.replace(temporary, path)
        if digest(path.read_bytes()) != digest(data):
            raise IOError("The saved file could not be verified; retry the operation.")
    finally:
        temporary.unlink(missing_ok=True)


def write_json(path, value):
    write_verified(path, json.dumps(value, ensure_ascii=False, indent=2).encode("utf-8"))


def root_path(relative):
    """Keep all dataset references relative to one portable dataset root."""
    result = (DATASET_ROOT / relative).resolve()
    if not result.is_relative_to(DATASET_ROOT.resolve()):
        raise ValueError("Invalid dataset-relative path.")
    return result


def initialise_dataset():
    for split in SPLITS:
        for folder in ("images", "records"):
            (DATASET_ROOT / split / folder).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "originals").mkdir(exist_ok=True)
    (DATASET_ROOT / "metadata").mkdir(exist_ok=True)
    settings = DATASET_ROOT / "metadata" / "dataset.json"
    if settings.exists():
        existing = json.loads(settings.read_text(encoding="utf-8"))
        if existing.get("schema_version") != SCHEMA_VERSION:
            raise ValueError("This dataset uses a different format. Choose a new dataset root.")
    else:
        write_json(settings, {"schema_version": SCHEMA_VERSION, "created_at": utc_now(),
                              "purpose": "Personal handwriting adaptation",
                              "validation_fraction": 0.25, "split_seed": 42,
                              "test_policy": "Reserved for separate teammate data; collector never assigns test."})
    readme = DATASET_ROOT / "README.txt"
    if not readme.exists():
        write_verified(readme, (
            "CVDL personal handwriting dataset\n"
            "Only human-approved crops and transcriptions are indexed.\n"
            "pending: approved data awaiting a 75/25 train/validation page split.\n"
            "test: reserved for teammates absent from personal fine-tuning.\n"
            "Use labels.csv, not a directory scan, to load training images.\n"
            "image_path and original_path are relative to this dataset root.\n"
            "Crop coordinates refer to the EXIF-upright RGB image; originals retain their bytes.\n"
            "PNG crops retain colour. Preprocessing/resizing happens in the training notebook.\n"
            "records/*.json are the recovery source; CSV and captures.json can be rebuilt.\n"
            "Run only one collection/split session against this root at a time.\n"
            "Finish collection and freeze a dataset version before comparing models.\n"
        ).encode("utf-8"))


def decode_capture(data_url):
    """Preserve the original bytes and use upright pixels for boxes and crops."""
    if not isinstance(data_url, str) or ";base64," not in data_url:
        raise ValueError("Choose a JPG, PNG or WebP image, or capture a photograph.")
    encoded = data_url.split(";base64,", 1)[1]
    if len(encoded) > 56 * 1024 * 1024:
        raise ValueError("Please use a photograph smaller than 40 MB.")
    raw = base64.b64decode(encoded, validate=True)
    with Image.open(io.BytesIO(raw)) as original:
        extension = {"JPEG": ".jpg", "PNG": ".png", "WEBP": ".webp"}.get(original.format)
        if extension is None or getattr(original, "n_frames", 1) != 1:
            raise ValueError("Use a still JPG, PNG or WebP image.")
        if original.width * original.height > 32_000_000:
            raise ValueError("This image exceeds 32 megapixels. Upload a paragraph crop or a smaller photograph.")
        upright = ImageOps.exif_transpose(original)
        if upright.mode in ("RGBA", "LA") or "transparency" in upright.info:
            rgba = upright.convert("RGBA")
            white = Image.new("RGBA", rgba.size, "white")
            upright = Image.alpha_composite(white, rgba).convert("RGB")
        else:
            upright = upright.convert("RGB")
        upright.load()
    return raw, upright, extension


def checked_box(box, size):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        raise ValueError("A crop must have left, top, right and bottom coordinates.")
    if not all(isinstance(x, (int, float)) and math.isfinite(x) for x in box):
        raise ValueError("Crop coordinates must be finite numbers.")
    width, height = size
    x1, y1, x2, y2 = [int(round(x)) for x in box]
    if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
        raise ValueError("Keep the complete crop inside the photograph.")
    if x2 - x1 < 3 or y2 - y1 < 3:
        raise ValueError("The crop is too small. Draw around a complete text line.")
    return [x1, y1, x2, y2]


def read_plan():
    path = DATASET_ROOT / "metadata" / "split_assignments.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {
        "schema_version": SCHEMA_VERSION, "seed": 42, "validation_fraction": 0.25, "pages": {}}


def committed_records():
    """A final record is the commit marker; unfinished image copies are ignored."""
    records = {}
    for split in SPLITS:
        for path in sorted((DATASET_ROOT / split / "records").glob("*.json")):
            record = json.loads(path.read_text(encoding="utf-8"))
            if record["split"] != split:
                raise ValueError(f"Record has the wrong split: {path.name}")
            previous = records.get(record["capture_id"])
            if previous:
                if previous["split"] != "pending":
                    raise ValueError("A capture appears in multiple final splits. Stop and inspect its records.")
                if previous["approval_hash"] != record["approval_hash"]:
                    raise ValueError("Conflicting capture records were found.")
            records[record["capture_id"]] = record
    return list(records.values())


def build_indexes():
    """Rebuild UTF-8 CSV files from saved records, including after a retry."""
    records = committed_records()
    for split in SPLITS:
        # Teammate test data is maintained separately: never rewrite its existing CSV.
        if split == "test" and (DATASET_ROOT / split / "labels.csv").exists():
            continue
        buffer = io.StringIO(newline="")
        writer = csv.DictWriter(buffer, fieldnames=LABEL_FIELDS)
        writer.writeheader()
        for record in sorted(records, key=lambda r: r["capture_id"]):
            if record["split"] == split:
                for line in record["lines"]:
                    writer.writerow({k: line[k] for k in LABEL_FIELDS})
        write_verified(DATASET_ROOT / split / "labels.csv", buffer.getvalue().encode("utf-8"))
    write_json(DATASET_ROOT / "captures.json", records)
    return records


def dataset_status():
    records = committed_records()
    counts = {split: {"pages": 0, "lines": 0, "en": 0, "zh": 0} for split in SPLITS}
    page_names = {}
    for split in SPLITS:
        subset = [r for r in records if r["split"] == split]
        counts[split]["pages"] = len({r["source_page_id"] for r in subset})
        for record in subset:
            count = len(record["lines"])
            counts[split]["lines"] += count
            counts[split][record["language"]] += count
            if split != "test":
                page_names[record["source_page_id"]] = record["page_label"]
    test_csv = DATASET_ROOT / "test" / "labels.csv"
    if test_csv.exists():
        with test_csv.open(encoding="utf-8-sig", newline="") as stream:
            test_rows = list(csv.DictReader(stream))
        counts["test"] = {"pages": len({r["source_page_id"] for r in test_rows if r.get("source_page_id")}),
                          "lines": len(test_rows),
                          "en": sum(r.get("language") == "en" for r in test_rows),
                          "zh": sum(r.get("language") == "zh" for r in test_rows)}
    return {"counts": counts, "pages": [{"id": p, "label": name} for p, name in sorted(page_names.items())],
            "root": str(DATASET_ROOT), "language": LANGUAGE}


## Cell 4: Save approved images and corrected labels to Drive

A capture record is written only after its original and approved crops have been saved. Retrying the same save does not create duplicate samples.

In [ ]:
# Cell 4: Save approved images and corrected labels to Drive

def save_approved_capture(capture, approved_lines, approved):
    """Commit complete image/label pairs and make retries idempotent."""
    if approved is not True:
        raise ValueError("Confirm that you checked every included crop and transcription.")
    if not approved_lines:
        raise ValueError("Include at least one correctly labelled line.")
    cleaned = []
    for row in approved_lines:
        text = str(row.get("text", "")).strip()
        if not text or "\n" in text or "\r" in text:
            raise ValueError("Each included crop needs one non-empty, single-line transcription.")
        cleaned.append({"box": checked_box(row["box"], capture["image"].size), "text": text,
                        "suggested_text": str(row.get("suggested_text", ""))})
    cleaned.sort(key=lambda row: (row["box"][1], row["box"][0]))
    approval_hash = digest(json.dumps(cleaned, ensure_ascii=False, sort_keys=True).encode())
    capture_id = capture["capture_id"]
    with DATA_LOCK:
        existing = next((r for r in committed_records() if r["capture_id"] == capture_id), None)
        if existing:
            if existing["approval_hash"] != approval_hash:
                raise ValueError("This capture was already saved with different labels. Its saved record was preserved.")
            # A failed index update can be completed without duplicating any samples.
            build_indexes()
            return {"message": "Already saved; dataset indexes verified.", "capture_id": capture_id,
                    "split": existing["split"], "lines": len(existing["lines"]), "status": dataset_status()}

        split = read_plan()["pages"].get(capture["source_page_id"], "pending")
        if split not in ("pending", "train", "validation"):
            raise ValueError("This collector is for your handwriting only. Teammate test pages are reserved separately.")
        original_path = f"originals/{capture_id}{capture['extension']}"
        write_verified(root_path(original_path), capture["raw"])
        lines = []
        for index, row in enumerate(cleaned, 1):
            sample_id = f"{capture_id}_L{index:03d}"
            image_path = f"{split}/images/{sample_id}.png"
            buffer = io.BytesIO()
            capture["image"].crop(row["box"]).save(buffer, format="PNG")
            data = buffer.getvalue()
            write_verified(root_path(image_path), data)
            lines.append({"sample_id": sample_id, "image_path": image_path, "text": row["text"],
                          "language": capture["language"], "source_page_id": capture["source_page_id"],
                          "capture_id": capture_id, "line_index": index, "box": row["box"],
                          "image_sha256": digest(data), "suggested_text": row["suggested_text"]})
        record = {"schema_version": SCHEMA_VERSION, "capture_id": capture_id,
                  "source_page_id": capture["source_page_id"], "page_label": capture["page_label"],
                  "language": capture["language"], "split": split, "approved_at": utc_now(),
                  "input_source": capture["input_source"], "input_mode": capture["input_mode"],
                  "original_filename": capture["filename"], "original_path": original_path,
                  "original_sha256": digest(capture["raw"]), "upright_size": list(capture["image"].size),
                  "coordinate_system": "EXIF-upright RGB pixels: left, top, right, bottom",
                  "recogniser": capture["recogniser"], "approval_hash": approval_hash, "lines": lines}
        # Write the commit marker last: partially uploaded crops never enter a label CSV.
        write_json(DATASET_ROOT / split / "records" / f"{capture_id}.json", record)
        try:
            build_indexes()
            message = f"Saved {len(lines)} approved line(s) to {split}."
        except Exception:
            message = (f"Saved {len(lines)} approved line(s). The export index needs rebuilding; "
                       "use Refresh / verify before training.")
        return {"message": message, "capture_id": capture_id, "split": split,
                "lines": len(lines), "status": dataset_status()}


## Cell 5: Prepare reproducible page-based training and validation splits

New pages initially go to `pending/`. The final interface provides a split button; this cell only defines the splitting and verification functions.

In [ ]:
# Cell 5: Prepare reproducible page-based training and validation splits

def verify_dataset():
    page_splits = defaultdict(set)
    for record in committed_records():
        page_splits[record["source_page_id"]].add(record["split"])
        if digest(root_path(record["original_path"]).read_bytes()) != record["original_sha256"]:
            raise ValueError("An original image is missing or changed: " + record["capture_id"])
        for line in record["lines"]:
            data = root_path(line["image_path"]).read_bytes()
            if digest(data) != line["image_sha256"] or not line["text"].strip():
                raise ValueError("A crop or label failed verification: " + line["sample_id"])
            with Image.open(io.BytesIO(data)) as image:
                image.verify()
    for splits in page_splits.values():
        if len(splits - {"pending"}) > 1:
            raise ValueError("A source page appears in more than one final split.")
    # A separately supplied test manifest is read-only to this collector.
    test_csv = DATASET_ROOT / "test" / "labels.csv"
    if test_csv.exists():
        with test_csv.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                if not row.get("text", "").strip() or not root_path(row["image_path"]).is_file():
                    raise ValueError("A separately supplied test image or label is missing.")
                if row.get("source_page_id") in page_splits and page_splits[row["source_page_id"]] - {"test"}:
                    raise ValueError("A test source page is also present in personal data.")
    return True


def prepare_training_split():
    """Freeze page assignments; later runs add new pages without reshuffling old ones."""
    with DATA_LOCK:
        verify_dataset()
        records = committed_records()
        personal = [r for r in records if r["split"] != "test"]
        pages = sorted({r["source_page_id"] for r in personal})
        if len(pages) < 2:
            raise ValueError("Collect at least two different source pages before creating a train/validation split.")
        plan = read_plan()
        # Reconstruct assignments from committed records if an older plan needs recovery.
        for record in personal:
            if record["split"] in ("train", "validation"):
                old = plan["pages"].get(record["source_page_id"], record["split"])
                if old != record["split"]:
                    raise ValueError("A saved page conflicts with the frozen split plan.")
                plan["pages"][record["source_page_id"]] = record["split"]
        new_pages = [page for page in pages if page not in plan["pages"]]
        # Use stable hashes as random keys: reproducible regardless of directory listing order.
        new_pages.sort(key=lambda page: digest(f"{plan['seed']}:{page}".encode()))
        target_validation = max(1, min(len(pages) - 1, round(len(pages) * 0.25)))
        existing_validation = sum(plan["pages"].get(page) == "validation" for page in pages)
        needed = max(0, min(len(new_pages), target_validation - existing_validation))
        for index, page in enumerate(new_pages):
            plan["pages"][page] = "validation" if index < needed else "train"
        write_json(DATASET_ROOT / "metadata" / "split_assignments.json", plan)

        # Copy and verify before removing pending copies; resume uses the same frozen plan.
        for record in personal:
            if record["split"] != "pending":
                continue
            destination = plan["pages"][record["source_page_id"]]
            updated = json.loads(json.dumps(record))
            updated["split"] = destination
            for line in updated["lines"]:
                old_path = line["image_path"]
                new_path = f"{destination}/images/{line['sample_id']}.png"
                write_verified(root_path(new_path), root_path(old_path).read_bytes())
                line["image_path"] = new_path
            write_json(DATASET_ROOT / destination / "records" / f"{record['capture_id']}.json", updated)
            (DATASET_ROOT / "pending" / "records" / f"{record['capture_id']}.json").unlink(missing_ok=True)
            for line in record["lines"]:
                root_path(line["image_path"]).unlink(missing_ok=True)

        # Clean harmless duplicate pending records left by a stopped previous transfer.
        for record in committed_records():
            if record["split"] != "pending":
                old_record = DATASET_ROOT / "pending" / "records" / f"{record['capture_id']}.json"
                if old_record.exists():
                    old = json.loads(old_record.read_text(encoding="utf-8"))
                    old_record.unlink()
                    for line in old["lines"]:
                        root_path(line["image_path"]).unlink(missing_ok=True)
        build_indexes()
        verify_dataset()
        status = dataset_status()
        warnings = []
        for language in ("en", "zh"):
            train_n = status["counts"]["train"][language]
            val_n = status["counts"]["validation"][language]
            if train_n + val_n and (not train_n or not val_n):
                warnings.append(f"{language.upper()} has no samples in one split; collect more independent pages before training that language.")
        return {"message": "Dataset exported and verified. Existing split assignments were retained.",
                "warnings": warnings, "status": status}


## Cell 6: Load recognisers on demand and detect horizontal text lines

The detector proposes line boxes. The recogniser processes one crop at a time. Only one language recogniser is retained in memory.

In [ ]:
# Cell 6: Load recognisers on demand and detect horizontal text lines

import gc
import time
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from transformers import (AutoImageProcessor, AutoModelForObjectDetection,
                          AutoModelForTextRecognition, RobertaTokenizer,
                          TrOCRProcessor, ViTImageProcessor, VisionEncoderDecoderModel)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENGLISH_MODEL_ID = "microsoft/trocr-base-handwritten"
CHINESE_MODEL_ID = "PaddlePaddle/PP-OCRv5_server_rec_safetensors"
DETECTOR_MODEL_ID = "PaddlePaddle/PP-OCRv5_mobile_det_safetensors"
MODELS = {"language": None, "recogniser": None, "processor": None,
          "detector": None, "detector_processor": None}


def load_recogniser(language):
    """Retain only one language recogniser to keep memory use bounded."""
    if MODELS["language"] == language and MODELS["recogniser"] is not None:
        return MODELS["recogniser"], MODELS["processor"]
    MODELS.update(language=None, recogniser=None, processor=None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if language == "en":
        # Match the tokenizer construction verified in the transcription prototype.
        tokenizer = RobertaTokenizer(
            vocab=hf_hub_download(ENGLISH_MODEL_ID, "vocab.json"),
            merges=hf_hub_download(ENGLISH_MODEL_ID, "merges.txt"), model_max_length=512)
        if tokenizer.encode("Hello world") != [0, 31414, 232, 2]:
            raise RuntimeError("English tokenizer verification failed. Restart the session and rerun Cell 1.")
        processor = TrOCRProcessor(
            image_processor=ViTImageProcessor.from_pretrained(ENGLISH_MODEL_ID), tokenizer=tokenizer)
        model = VisionEncoderDecoderModel.from_pretrained(ENGLISH_MODEL_ID, dtype=torch.float32)
    elif language == "zh":
        processor = AutoImageProcessor.from_pretrained(CHINESE_MODEL_ID)
        model = AutoModelForTextRecognition.from_pretrained(CHINESE_MODEL_ID, dtype=torch.float32)
    else:
        raise ValueError("Select English or Chinese in Cell 2.")
    model = model.to(DEVICE).eval()
    MODELS.update(language=language, recogniser=model, processor=processor)
    return model, processor


def load_detector():
    # A small CPU detector leaves GPU memory available for the recogniser.
    if MODELS["detector"] is None:
        processor = AutoImageProcessor.from_pretrained(DETECTOR_MODEL_ID)
        model = AutoModelForObjectDetection.from_pretrained(DETECTOR_MODEL_ID, dtype=torch.float32).cpu().eval()
        MODELS.update(detector=model, detector_processor=processor)
    return MODELS["detector"], MODELS["detector_processor"]


def merge_horizontal_fragments(boxes):
    """Join same-row word fragments for single-column, roughly horizontal notes."""
    rows = []
    for box in sorted(boxes, key=lambda b: ((b[1] + b[3]) / 2, b[0])):
        best = None
        for row in rows:
            a, b = row, box
            overlap = min(a[3], b[3]) - max(a[1], b[1])
            minimum_height = min(a[3] - a[1], b[3] - b[1])
            centres = abs((a[1] + a[3] - b[1] - b[3]) / 2)
            if overlap >= 0.55 * minimum_height and centres <= 0.55 * minimum_height:
                best = row
                break
        if best is None:
            rows.append(list(box))
        else:
            best[:] = [min(best[0], box[0]), min(best[1], box[1]),
                       max(best[2], box[2]), max(best[3], box[3])]
    return sorted(rows, key=lambda b: (b[1], b[0]))


def detect_lines(image):
    model, processor = load_detector()
    # Limit detection cost only. The saved crops still use the full-resolution image.
    scale = min(1.0, 1600 / max(image.size))
    small = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))))
    with torch.inference_mode():
        inputs = processor(images=small, return_tensors="pt")
        outputs = model(**inputs)
        result = processor.post_process_object_detection(outputs, target_sizes=inputs["target_sizes"])[0]
    raw_boxes = result["boxes"]
    if hasattr(raw_boxes, "detach"):
        raw_boxes = raw_boxes.detach().cpu().numpy()
    boxes = []
    for raw in raw_boxes:
        array = np.asarray(raw, dtype=float)
        if array.size == 4:
            x1, y1, x2, y2 = array.reshape(-1).tolist()
        else:
            points = array.reshape(-1, 2)
            x1, y1 = points.min(axis=0)
            x2, y2 = points.max(axis=0)
        box = [max(0, math.floor(x1 / scale)), max(0, math.floor(y1 / scale)),
               min(image.width, math.ceil(x2 / scale)), min(image.height, math.ceil(y2 / scale))]
        if box[2] - box[0] >= 3 and box[3] - box[1] >= 3:
            boxes.append(box)
    boxes = merge_horizontal_fragments(boxes)
    padded = []
    for x1, y1, x2, y2 in boxes:
        padding = max(2, round((y2 - y1) * 0.08))
        padded.append([max(0, x1 - padding), max(0, y1 - padding),
                       min(image.width, x2 + padding), min(image.height, y2 + padding)])
    if len(padded) > 80:
        raise ValueError("Too many regions were detected. Use a closer paragraph photo, or draw the required lines manually.")
    return padded


def transcribe_crop(image, language):
    model, processor = load_recogniser(language)
    image = image.convert("RGB")
    with torch.inference_mode():
        if language == "en":
            pixels = processor(images=image, return_tensors="pt").pixel_values.to(DEVICE)
            ids = model.generate(pixels, max_new_tokens=160)
            return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        # Keep extreme crop widths from inflating Chinese recognition memory.
        if image.width / image.height > 45:
            raise ValueError("This crop is unusually wide. Check that it contains one complete line.")
        inputs = processor(images=image, return_tensors="pt").to(DEVICE)
        outputs = model(**inputs)
        return processor.post_process_text_recognition(outputs)[0]["text"].strip()


## Cell 7: Initialise storage and download the selected models once

Wait for model loading to finish. If a download fails, rerun this cell to retry. You can still use manual crop drawing and text entry in the collection interface.

In [ ]:
# Cell 7: Initialise storage and download the selected models once

initialise_dataset()
build_indexes()
print(f"Dataset: {DATASET_ROOT}")
print(f"Language: {LANGUAGE.upper()} | Recogniser device: {DEVICE}")
if DEVICE == "cpu":
    print("CPU mode is supported, but OCR is slower. A Colab T4 GPU is recommended.")
for label, loader in [("text detector", load_detector),
                      (f"{LANGUAGE.upper()} recogniser", lambda: load_recogniser(LANGUAGE))]:
    print(f"Loading {label}; the first run downloads pretrained weights...")
    try:
        started = time.perf_counter()
        loader()
        print(f"Ready: {label} ({time.perf_counter() - started:.1f}s).")
    except Exception as exc:
        print(f"{label} could not load: {type(exc).__name__}: {str(exc)[:400]}")
        print("You can rerun this cell to retry. Manual crop drawing and labelling remain available.")
print("Run Cells 8 and 9 to open the collection interface.")


## Cell 8: Connect the collection interface to Python callbacks

Registers the actions used by the interface. Progress is shown one line at a time, and recoverable failures appear as messages beside the review.

In [ ]:
# Cell 8: Connect the collection interface to Python callbacks

from google.colab import output
from IPython.display import JSON

ACTIVE_CAPTURE = None


def begin_capture(payload):
    global ACTIVE_CAPTURE
    raw, image, extension = decode_capture(payload["data_url"])
    mode = payload.get("mode", "multiple")
    if mode not in ("single", "multiple"):
        raise ValueError("Choose single-line or multiple-line mode.")
    pages = {p["id"]: p["label"] for p in dataset_status()["pages"]}
    page_id = payload.get("page_id", "")
    if page_id and page_id not in pages:
        raise ValueError("That source page is unavailable. Refresh the page list.")
    if not page_id:
        page_id = "page_" + uuid.uuid4().hex
    capture_id = "cap_" + uuid.uuid4().hex
    page_label = pages.get(page_id) or str(payload.get("page_label", "")).strip()[:120]
    if not page_label:
        page_label = "Page " + page_id[-8:]
    ACTIVE_CAPTURE = {"capture_id": capture_id, "raw": raw, "image": image, "extension": extension,
                      "source_page_id": page_id, "page_label": page_label, "language": LANGUAGE,
                      "filename": str(payload.get("filename", "camera.jpg"))[:200],
                      "input_source": "camera" if payload.get("source") == "camera" else "upload",
                      "input_mode": mode,
                      "recogniser": ENGLISH_MODEL_ID if LANGUAGE == "en" else CHINESE_MODEL_ID}
    warning = ""
    if mode == "single":
        boxes = [[0, 0, image.width, image.height]]
    else:
        try:
            boxes = detect_lines(image)
            if not boxes:
                warning = "No lines were detected. Use Add line to draw a complete line manually."
        except Exception as exc:
            boxes = []
            warning = f"Automatic detection is unavailable: {str(exc)[:220]}. You can draw line boxes manually."
    # Use an EXIF-upright preview so browser and Python always agree on coordinates.
    preview = image.copy()
    preview.thumbnail((2400, 2400))
    buffer = io.BytesIO()
    preview.save(buffer, format="JPEG", quality=94)
    return {"capture_id": capture_id, "source_page_id": page_id, "page_label": page_label,
            "width": image.width, "height": image.height, "boxes": boxes, "warning": warning,
            "preview": "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()}


def current_capture(capture_id):
    if ACTIVE_CAPTURE is None or ACTIVE_CAPTURE["capture_id"] != capture_id:
        raise ValueError("This capture is no longer active. Analyse the image again before continuing.")
    return ACTIVE_CAPTURE


def recognise_line(payload):
    capture = current_capture(payload["capture_id"])
    box = checked_box(payload["box"], capture["image"].size)
    started = time.perf_counter()
    text = transcribe_crop(capture["image"].crop(box), capture["language"])
    return {"text": text, "seconds": round(time.perf_counter() - started, 2)}


def save_from_interface(payload):
    capture = current_capture(payload["capture_id"])
    return save_approved_capture(capture, payload["lines"], payload.get("approved", False))


def refresh_dataset():
    with DATA_LOCK:
        verify_dataset()
        build_indexes()
        return dataset_status()


def register_collector_callback(name, function):
    # Convert expected input/download failures into readable UI feedback.
    def callback(*args):
        try:
            return JSON({"ok": True, "data": function(*args)})
        except Exception as exc:
            return JSON({"ok": False, "error": f"{type(exc).__name__}: {str(exc)[:700]}"})
    output.register_callback("cvdl.collector." + name, callback)


for name, function in [("begin", begin_capture), ("ocr", recognise_line),
                       ("save", save_from_interface), ("status", refresh_dataset),
                       ("split", prepare_training_split)]:
    register_collector_callback(name, function)
print("Collection callbacks registered. Run Cell 9 to open the interface.")


## Cell 9: Open the camera/upload review and saving interface

**Collection steps**

1. Choose **Open camera** or **Upload image**. Preview the image; capture again or upload a replacement if needed.
2. Choose single-line or multiple-line mode. For a new physical page, keep **New physical page** and optionally describe it. For another paragraph/photograph of that page, select its existing entry after choosing the image.
3. Select **Detect and transcribe**. Review all proposed boxes and their crop previews.
4. Correct the text. Use **Add line**, **Redraw crop**, the coordinate fields, and inclusion checkboxes as needed. A changed crop clears its earlier text. Use **Transcribe** or type the correct label.
5. Confirm you checked the included crops and labels; select **Approve and save to Drive**. Wait for the saved confirmation before selecting another image.
6. After collecting at least two different source pages, select **Prepare 75/25 training split**. Four or more pages give a more useful starting split; collecting many more independent pages is preferable. Actual line ratios can differ from page ratios.

**Files and training handoff**

| Location relative to the dataset folder | Contents |
|---|---|
| `originals/` | Original camera/upload bytes |
| `pending/images/`, `pending/labels.csv` | Approved crops awaiting a split |
| `train/images/`, `train/labels.csv` | Training crops and corrected labels |
| `validation/images/`, `validation/labels.csv` | Validation crops and corrected labels |
| `test/images/`, `test/labels.csv` | Reserved, initially empty teammate test set |
| Each split's `records/` | One recoverable JSON record per saved capture |
| `metadata/split_assignments.json` | Frozen source-page assignments |
| `captures.json` | Rebuilt index of capture metadata and crop coordinates |

Each UTF-8 label CSV contains `sample_id`, `image_path`, `text`, `language`, `source_page_id`, `capture_id`, `line_index`. Paths are relative to the dataset root. In the training notebook, filter by `language`, open `DATASET_ROOT / image_path` with Pillow, and use `text` as the reference. Its dataset-loading cell will need that small adapter; this collector does not change the training notebook.

Only load entries listed in `labels.csv`. Unfinished copies are excluded until their record is committed. Saved images use full-resolution upright pixels; on-screen previews may be smaller. Keep dataset versions fixed for model comparisons. This is line-recognition data: evaluating automatic paragraph detection or reading order is a separate evaluation.

**Recovery:** if a save reports a failure, keep the review open and retry. **Refresh / verify** rebuilds CSV indexes from saved records. A disconnected Colab runtime can lose the current unsaved capture; all confirmed saves remain in Drive. Reconnect, rerun the setup cells, and upload/capture that unsaved image again. Camera permission or browser problems can be handled using image upload.

**Scope:** this version uses Google Drive. OneDrive integration, batch image imports, translation and training are not implemented. Review and save one image at a time. Teammate test data should be labelled separately and remain excluded from tuning.

**Model references:** [TrOCR handwritten](https://huggingface.co/microsoft/trocr-base-handwritten), [PP-OCRv5 mobile detector](https://huggingface.co/PaddlePaddle/PP-OCRv5_mobile_det_safetensors), [PP-OCRv5 server recogniser](https://huggingface.co/PaddlePaddle/PP-OCRv5_server_rec_safetensors). [Colab storage and runtime guidance](https://research.google.com/colaboratory/faq.html).


In [ ]:
# Cell 9: Open the camera/upload review and saving interface
from IPython.display import Javascript, display

# All UI actions use the registered callbacks; no training starts here.
COLLECTION_UI = r'''(async () => {
  const initial = __INITIAL__;
  const root = document.createElement('div');
  root.className = 'cvdl-collector';
  root.innerHTML = `
  <style>
  .cvdl-collector{font:14px/1.5 system-ui,sans-serif;color:#172338;background:#f6f8fc;padding:22px;border:1px solid #d9e1ed;border-radius:16px;max-width:1100px;margin:auto}
  .cvdl-collector *{box-sizing:border-box}.cvdl-collector h2{font-size:25px;margin:0 0 4px}.cvdl-collector h3{font-size:17px;margin:0 0 12px}
  .cvdl-collector p{margin:6px 0 14px}.cvdl-collector .muted{color:#56657b}.cvdl-collector .panel{background:white;border:1px solid #dde4ef;border-radius:12px;padding:18px;margin-top:16px}
  .cvdl-collector .toolbar{display:flex;gap:9px;flex-wrap:wrap;align-items:center}.cvdl-collector button{border:1px solid #bac9dd;border-radius:7px;background:white;padding:9px 13px;color:#1b385d;cursor:pointer;font:inherit}
  .cvdl-collector button.primary{background:#2355b5;color:white;border-color:#2355b5}.cvdl-collector button:disabled{opacity:.45;cursor:default}
  .cvdl-collector input[type=text],.cvdl-collector select{border:1px solid #bdcadb;border-radius:6px;padding:9px;font:inherit;max-width:100%;background:white;color:#172338}
  .cvdl-collector label.field{display:flex;flex-direction:column;gap:5px;flex:1;min-width:180px}.cvdl-collector .status{padding:12px 15px;background:#e9f0fc;border-radius:9px;margin-top:14px;white-space:pre-wrap;overflow-wrap:anywhere}
  .cvdl-collector .status.error{background:#fff0eb;color:#873615}.cvdl-collector .status.success{background:#e7f5ed;color:#215e3c}
  .cvdl-collector video,.cvdl-collector .source-preview{max-width:100%;max-height:460px;margin:12px auto;display:block;border-radius:7px}
  .cvdl-collector [hidden]{display:none!important}.cvdl-collector .page-canvas{width:100%;height:auto;display:block;border:1px solid #ccd8e7;border-radius:6px;margin:12px 0;touch-action:pan-y}
  .cvdl-collector .line-card{border:1px solid #d7e0ec;border-radius:9px;margin:12px 0;padding:12px;background:#fbfcff}.cvdl-collector .line-card.selected{border:2px solid #2355b5;padding:11px}
  .cvdl-collector .line-card.excluded{opacity:.6}.cvdl-collector .line-preview{max-width:100%;height:auto;display:block;background:white;border:1px solid #e4e9f1;margin:10px 0}
  .cvdl-collector .line-text{width:100%;font-size:17px!important}.cvdl-collector .coords{display:flex;flex-wrap:wrap;gap:9px;margin-top:9px}.cvdl-collector .coords label{font-size:12px}
  .cvdl-collector input[type=number]{display:block;width:85px;padding:6px;border:1px solid #bdcadb;border-radius:4px}.cvdl-collector .stats{display:flex;flex-wrap:wrap;gap:12px;margin:12px 0}
  .cvdl-collector .stat{background:#eef3fa;padding:10px 15px;border-radius:8px;min-width:135px}.cvdl-collector .stat strong{font-size:20px;display:block}
  .cvdl-collector .check{display:flex;gap:8px;align-items:flex-start;margin:13px 0}.cvdl-collector .small{font-size:12px}.cvdl-collector details{margin-top:10px}
  </style>
  <h2>Handwriting collection</h2>
  <p class="muted">Capture or upload · Review every line · Save corrected examples</p>
  <div class="toolbar"><strong data-language></strong><span class="muted">Your handwriting → 75% training / 25% validation. Teammate test data stays separate.</span></div>
  <div class="panel">
    <h3>1. Choose an image</h3>
    <div class="toolbar">
      <label class="field">Image contains<select data-mode><option value="multiple">Multiple lines / a paragraph</option><option value="single">One text line</option></select></label>
      <label class="field">Source page<select data-page><option value="">New physical page</option></select></label>
      <label class="field">New page description (optional)<input data-page-label type="text" maxlength="120" placeholder="e.g. Embedded systems, page 3"></label>
    </div>
    <p class="small muted">For another paragraph or photograph of an existing page, select that same source page. Use one language and a single column per capture.</p>
    <div class="toolbar"><button data-camera>Open camera</button><button data-upload>Upload image</button><input data-file type="file" accept="image/jpeg,image/png,image/webp" hidden><button data-stop hidden>Close camera</button></div>
    <video data-video autoplay playsinline muted hidden></video>
    <button class="primary" data-capture hidden>Capture photograph</button>
    <img class="source-preview" data-preview alt="Selected photograph" hidden>
    <div class="toolbar"><button class="primary" data-analyse>Detect and transcribe</button><span class="small muted" data-filename>No image selected.</span></div>
  </div>
  <div class="status" data-status role="status" aria-live="polite">Choose a camera photograph or an existing image to begin.</div>
  <div class="panel" data-review hidden>
    <h3>2. Check line boundaries and correct the text</h3>
    <p class="muted">Check that every box contains exactly one complete line. Add missed lines; redraw or exclude incorrect boxes. Type what is written, preserving spelling and symbols.</p>
    <div class="toolbar"><button data-add>Add line</button><button data-cancel-draw hidden>Cancel drawing</button><button data-transcribe>Transcribe empty lines</button><span data-draw-hint class="small muted"></span></div>
    <canvas data-canvas class="page-canvas" aria-label="Source photograph with numbered line boxes"></canvas>
    <div data-lines></div>
    <label class="check"><input data-approved type="checkbox"><span>I checked every included crop and its transcription. These labels reflect the handwriting in the images.</span></label>
    <button data-save class="primary">Approve and save to Drive</button>
    <p class="small muted">Only included, non-empty lines are saved. Changing a crop clears its old transcription so an outdated label cannot be saved accidentally.</p>
  </div>
  <div class="panel">
    <h3>3. Dataset progress</h3><div class="stats" data-stats></div>
    <div class="toolbar"><button data-refresh>Refresh / verify</button><button data-split>Prepare 75/25 training split</button></div>
    <p class="small muted">Collect first, then prepare the split. Whole source pages stay together. Existing assignments remain fixed; line percentages can differ from 75/25.</p>
    <p class="small muted" data-root></p>
  </div>`;
  document.body.appendChild(root);
  // Let Colab resize the output frame as crops and editable rows are added.
  if(typeof ResizeObserver !== 'undefined') new ResizeObserver(()=>{
    globalThis.google?.colab?.output?.setIframeHeight?.(document.documentElement.scrollHeight,true);
  }).observe(root);
  const $ = selector => root.querySelector(selector);
  const state = {busy:false, saved:false, raw:null, filename:'', source:'upload', info:null,
                 rows:[], image:null, stream:null, drawMode:null, drag:null, selected:null};
  const video = $('[data-video]'), canvas = $('[data-canvas]');
  const context = canvas.getContext('2d');
  $('[data-language]').textContent = initial.language === 'zh' ? 'Chinese' : 'English';

  function message(text, kind='') {
    $('[data-status]').textContent = text;
    $('[data-status]').className = 'status ' + kind;
  }
  function controls() {
    root.querySelectorAll('button,input,select').forEach(el => {el.disabled=state.busy;});
    $('[data-analyse]').disabled = state.busy || !state.raw;
    $('[data-add]').disabled = $('[data-transcribe]').disabled = state.busy || !state.info || state.saved;
    const included=state.rows.filter(row=>row.include);
    $('[data-save]').disabled = state.busy || state.saved || !state.info || !$('[data-approved]').checked ||
      !included.length || included.some(row=>!row.text.trim());
    if(state.saved) $('[data-review]').querySelectorAll('input,button').forEach(el=>{el.disabled=true;});
    $('[data-mode]').disabled = $('[data-page]').disabled = $('[data-page-label]').disabled = state.busy || !!state.info;
    $('[data-page-label]').disabled ||= !!$('[data-page]').value;
  }
  async function request(name, ...args) {
    const response = await google.colab.kernel.invokeFunction('cvdl.collector.'+name,args,{});
    let result = response.data?.['application/json'];
    if(typeof result === 'string') result=JSON.parse(result);
    if(!result) throw new Error('No response from Python. Reconnect the runtime, then rerun Cells 7–9.');
    if(!result.ok) throw new Error(result.error);
    return result.data;
  }
  async function task(fn) {
    if(state.busy) return;
    state.busy=true;controls();
    try {await fn();} catch(error){message(error.message+' Your current review has been kept; correct the issue and retry.','error');}
    finally {state.busy=false;controls();}
  }
  function updateStatus(data) {
    const selection=$('[data-page]').value;
    $('[data-page]').replaceChildren(new Option('New physical page',''));
    for(const page of data.pages) $('[data-page]').add(new Option(page.label+' · '+page.id.slice(-6),page.id));
    $('[data-page]').value=selection;
    $('[data-stats]').replaceChildren();
    for(const [key,label] of [['pending','Awaiting split'],['train','Training'],['validation','Validation'],['test','Reserved test']]) {
      const item=document.createElement('div');item.className='stat';
      const count=document.createElement('strong');count.textContent=data.counts[key].lines+' lines';
      const text=document.createElement('div');text.textContent=label+' · '+data.counts[key].pages+' pages';
      const languages=document.createElement('div');languages.className='small muted';languages.textContent='EN '+data.counts[key].en+' / ZH '+data.counts[key].zh;
      item.append(count,text,languages);$('[data-stats]').appendChild(item);
    }
    $('[data-root]').textContent='Saved dataset: '+data.root;
  }
  function stopCamera() {
    if(state.stream) state.stream.getTracks().forEach(track=>track.stop());
    state.stream=null;video.srcObject=null;video.hidden=true;
    $('[data-capture]').hidden=$('[data-stop]').hidden=true;
  }
  window.addEventListener('pagehide',stopCamera);
  $('[data-stop]').onclick=stopCamera;
  function mayReplace(){return !state.info || state.saved || confirm('Discard this unsaved review and choose another image?');}
  function loadImage(url){return new Promise((resolve,reject)=>{const img=new Image();img.onload=()=>resolve(img);img.onerror=()=>reject(new Error('The image could not be opened. Use a JPG, PNG or WebP photograph.'));img.src=url;});}
  async function selectImage(dataUrl, filename, source) {
    await loadImage(dataUrl);
    state.raw=dataUrl;state.filename=filename;state.source=source;state.info=null;state.rows=[];state.saved=false;state.drawMode=null;
    $('[data-approved]').checked=false;$('[data-review]').hidden=true;$('[data-preview]').src=dataUrl;$('[data-preview]').hidden=false;
    $('[data-filename]').textContent=filename;$('[data-page]').value='';$('[data-page-label]').value='';
    message('Image ready. Select its source page and line mode, then detect and transcribe.');controls();
  }
  $('[data-camera]').onclick=()=>task(async()=>{
    if(!mayReplace()) return;
    stopCamera();
    if(!navigator.mediaDevices?.getUserMedia) throw new Error('Camera access is unavailable in this browser. Use Upload image.');
    try {
      state.stream=await navigator.mediaDevices.getUserMedia({video:{width:{ideal:3840},height:{ideal:2160},facingMode:{ideal:'environment'}},audio:false});
      video.srcObject=state.stream;video.hidden=false;await video.play();
      $('[data-capture]').hidden=$('[data-stop]').hidden=false;
      message('Position one line or a paragraph in good lighting, then capture.');
    } catch(error) {stopCamera();throw new Error('Camera could not open. Allow camera permission or use Upload image. '+error.message);}
  });
  $('[data-capture]').onclick=()=>task(async()=>{
    if(!video.videoWidth) throw new Error('Wait for the camera preview before capturing.');
    const snapshot=document.createElement('canvas');snapshot.width=video.videoWidth;snapshot.height=video.videoHeight;
    snapshot.getContext('2d').drawImage(video,0,0);const data=snapshot.toDataURL('image/jpeg',0.95);stopCamera();
    await selectImage(data,'camera_'+Date.now()+'.jpg','camera');
  });
  $('[data-upload]').onclick=()=>{if(!state.busy&&mayReplace()){$('[data-file]').value='';$('[data-file]').click();}};
  $('[data-file]').onchange=()=>task(async()=>{
    const file=$('[data-file]').files[0];if(!file)return;
    if(file.size>40*1024*1024) throw new Error('Please select an image smaller than 40 MB.');
    const data=await new Promise((resolve,reject)=>{const reader=new FileReader();reader.onload=()=>resolve(reader.result);reader.onerror=reject;reader.readAsDataURL(file);});
    stopCamera();await selectImage(data,file.name,'upload');
  });
  $('[data-page]').onchange=controls;

  // Draw in preview pixels while retaining full-resolution coordinates in every row.
  function drawPage() {
    if(!state.info||!state.image)return;
    canvas.width=Math.min(1100,state.info.width);canvas.height=Math.round(canvas.width*state.info.height/state.info.width);
    context.fillStyle='white';context.fillRect(0,0,canvas.width,canvas.height);context.drawImage(state.image,0,0,canvas.width,canvas.height);
    const ratio=canvas.width/state.info.width;
    state.rows.forEach((row,index)=>{
      const [x1,y1,x2,y2]=row.box.map(x=>x*ratio);context.strokeStyle=row.include?'#175abd':'#88919d';context.lineWidth=row.id===state.selected?3:1.5;
      context.strokeRect(x1,y1,x2-x1,y2-y1);context.fillStyle=context.strokeStyle;context.fillRect(x1,Math.max(0,y1-20),26,20);
      context.fillStyle='white';context.font='bold 13px system-ui';context.fillText(String(index+1),x1+5,Math.max(14,y1-5));
    });
    if(state.drag){const [x1,y1]=state.drag.start,[x2,y2]=state.drag.end;context.strokeStyle='#d24420';context.lineWidth=2;context.strokeRect(x1*ratio,y1*ratio,(x2-x1)*ratio,(y2-y1)*ratio);}
    canvas.style.touchAction=state.drawMode?'none':'pan-y';canvas.style.cursor=state.drawMode?'crosshair':'default';
  }
  function thumbnail(box) {
    const out=document.createElement('canvas');const [x1,y1,x2,y2]=box;
    out.width=Math.min(1000,x2-x1);out.height=Math.max(1,Math.round(out.width*(y2-y1)/(x2-x1)));
    const sx=state.image.naturalWidth/state.info.width,sy=state.image.naturalHeight/state.info.height;
    const ctx=out.getContext('2d');ctx.fillStyle='white';ctx.fillRect(0,0,out.width,out.height);
    ctx.drawImage(state.image,x1*sx,y1*sy,(x2-x1)*sx,(y2-y1)*sy,0,0,out.width,out.height);return out.toDataURL('image/jpeg',0.94);
  }
  function startDrawing(id) {
    state.drawMode=id;state.selected=id==='new'?null:id;
    $('[data-cancel-draw]').hidden=false;$('[data-draw-hint]').textContent='Drag a rectangle around one complete line on the photograph.';drawPage();
    canvas.scrollIntoView({block:'center',behavior:'smooth'});
  }
  function endDrawing(){state.drawMode=null;state.drag=null;$('[data-cancel-draw]').hidden=true;$('[data-draw-hint]').textContent='';drawPage();}
  $('[data-add]').onclick=()=>startDrawing('new');$('[data-cancel-draw]').onclick=endDrawing;
  function cropChanged(row,box){row.box=box;row.text='';row.suggested_text='';$('[data-approved]').checked=false;state.rows.sort((a,b)=>a.box[1]-b.box[1]||a.box[0]-b.box[0]);renderLines();drawPage();controls();}
  function renderLines() {
    $('[data-lines]').replaceChildren();
    state.rows.forEach((row,index)=>{
      const card=document.createElement('div');card.className='line-card'+(row.include?'':' excluded')+(row.id===state.selected?' selected':'');
      const tools=document.createElement('div');tools.className='toolbar';
      const include=document.createElement('input');include.type='checkbox';include.checked=row.include;include.setAttribute('aria-label','Include line '+(index+1));
      include.onchange=()=>{row.include=include.checked;$('[data-approved]').checked=false;renderLines();drawPage();controls();};
      const label=document.createElement('strong');label.textContent='Line '+(index+1);
      const redraw=document.createElement('button');redraw.textContent='Redraw crop';redraw.onclick=()=>startDrawing(row.id);
      const ocr=document.createElement('button');ocr.textContent='Transcribe';ocr.onclick=()=>task(async()=>{if(row.text.trim()&&!confirm('Replace this line’s current text with a new model suggestion?'))return;await transcribeRow(row,index);message('Suggestion ready. Check it against the crop.');});
      tools.append(include,label,redraw,ocr);
      const preview=document.createElement('img');preview.className='line-preview';preview.src=thumbnail(row.box);preview.alt='Image crop for line '+(index+1);
      const text=document.createElement('input');text.type='text';text.className='line-text';text.value=row.text;text.placeholder='Type or correct the complete text in this crop';text.setAttribute('aria-label','Transcription for line '+(index+1));
      text.oninput=()=>{row.text=text.value;$('[data-approved]').checked=false;controls();};
      const details=document.createElement('details'),summary=document.createElement('summary');summary.textContent='Adjust crop coordinates';details.appendChild(summary);
      const coords=document.createElement('div');coords.className='coords';
      ['Left','Top','Right','Bottom'].forEach((name,i)=>{const lab=document.createElement('label');lab.textContent=name;const input=document.createElement('input');input.type='number';input.step='1';input.min=0;input.max=i%2?state.info.height:state.info.width;input.value=row.box[i];input.setAttribute('aria-label',name+' for line '+(index+1));input.onchange=()=>{const next=[...row.box];next[i]=Math.round(Number(input.value));if(!next.every(Number.isFinite)||next[0]<0||next[1]<0||next[2]>state.info.width||next[3]>state.info.height||next[2]-next[0]<3||next[3]-next[1]<3){input.value=row.box[i];message('Keep the crop inside the photograph and at least 3 pixels wide and tall.','error');return;}cropChanged(row,next);message('Crop updated. Transcribe it again or type its corrected label.');};lab.appendChild(input);coords.appendChild(lab);});
      details.appendChild(coords);card.append(tools,preview,text,details);$('[data-lines]').appendChild(card);
    });controls();
  }
  function point(event){const rect=canvas.getBoundingClientRect();return [Math.max(0,Math.min(state.info.width,Math.round((event.clientX-rect.left)/rect.width*state.info.width))),Math.max(0,Math.min(state.info.height,Math.round((event.clientY-rect.top)/rect.height*state.info.height)))];}
  canvas.onpointerdown=event=>{if(!state.drawMode||state.busy||state.saved)return;event.preventDefault();canvas.setPointerCapture(event.pointerId);state.drag={start:point(event),end:point(event)};drawPage();};
  canvas.onpointermove=event=>{if(state.drag){state.drag.end=point(event);drawPage();}};
  canvas.onpointerup=event=>{
    if(!state.drag)return;const [a,b]=state.drag.start,[c,d]=point(event);const box=[Math.min(a,c),Math.min(b,d),Math.max(a,c),Math.max(b,d)];
    if(box[2]-box[0]<3||box[3]-box[1]<3){state.drag=null;drawPage();return;}
    const id=state.drawMode;endDrawing();
    if(id==='new'){const row={id:crypto.randomUUID(),box,text:'',suggested_text:'',include:true};state.rows.push(row);cropChanged(row,box);}else cropChanged(state.rows.find(r=>r.id===id),box);
    message('Crop ready. Transcribe this line or type the label manually.');
  };
  canvas.onpointercancel=()=>{state.drag=null;drawPage();};

  // One callback per line exposes progress and keeps inference sequential.
  async function transcribeRow(row,index) {
    message('Transcribing line '+(index+1)+' of '+state.rows.length+'…');
    const data=await request('ocr',{capture_id:state.info.capture_id,box:row.box});
    row.text=data.text;row.suggested_text=data.text;$('[data-approved]').checked=false;renderLines();
  }
  async function transcribeEmpty() {
    let failed=0,lastError='';
    for(let i=0;i<state.rows.length;i++) if(state.rows[i].include&&!state.rows[i].text.trim()) {
      try{await transcribeRow(state.rows[i],i);}catch(error){failed++;lastError=error.message;break;}
    }
    if(failed)message('OCR could not finish. You can type labels manually, or retry Transcribe empty lines. '+lastError,'error');
    else message('Review every included crop and transcription, then approve and save.');
  }
  $('[data-transcribe]').onclick=()=>task(transcribeEmpty);
  $('[data-analyse]').onclick=()=>task(async()=>{
    if(state.info&&!state.saved&&!confirm('Replace this unsaved review by analysing the image again?'))return;
    message('Finding text lines…');stopCamera();
    const data=await request('begin',{data_url:state.raw,filename:state.filename,source:state.source,mode:$('[data-mode]').value,page_id:$('[data-page]').value,page_label:$('[data-page-label]').value});
    const preview=await loadImage(data.preview);state.info=data;state.image=preview;state.saved=false;state.selected=null;state.drawMode=null;state.drag=null;
    state.rows=data.boxes.map(box=>({id:crypto.randomUUID(),box,text:'',suggested_text:'',include:true}));
    $('[data-preview]').hidden=true;$('[data-review]').hidden=false;$('[data-approved]').checked=false;renderLines();drawPage();
    if(data.warning)message(data.warning,'error');else await transcribeEmpty();
  });
  $('[data-approved]').onchange=controls;
  $('[data-save]').onclick=()=>task(async()=>{
    message('Saving approved images and labels to Google Drive…');
    const data=await request('save',{capture_id:state.info.capture_id,approved:$('[data-approved]').checked,
      lines:state.rows.filter(r=>r.include).map(r=>({box:r.box,text:r.text,suggested_text:r.suggested_text}))});
    state.saved=true;updateStatus(data.status);message(data.message+' You can capture or upload the next image.','success');
  });
  $('[data-refresh]').onclick=()=>task(async()=>{message('Verifying saved images and labels…');updateStatus(await request('status'));message('Saved dataset verified.','success');});
  $('[data-split]').onclick=()=>task(async()=>{
    if(state.info&&!state.saved)throw new Error('Save or finish the current review before preparing the split.');
    message('Preparing the page-based split and verifying exported files…');
    const data=await request('split');updateStatus(data.status);message(data.message+(data.warnings.length?'\n'+data.warnings.join('\n'):''),data.warnings.length?'':'success');
  });
  updateStatus(initial);controls();
})();
'''
display(Javascript(COLLECTION_UI.replace("__INITIAL__", json.dumps(dataset_status(), ensure_ascii=False))))
